In [1]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from rank_bm25 import BM25Okapi
from langchain_core.documents import Document
from IPython.display import Markdown, display
import os

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_14668\3290983038.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader
d:\Gitlab\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load Documents from Directory

In [2]:
loader = DirectoryLoader(
    "../handbook",
    glob="*.md",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"}
)

documents = loader.load()
documents[0].page_content

"# Benefits & Perks\n\n## Health Insurance\n\nDetailed information about all 37signals insurance policies and other benefits can be found in [Basecamp](https://3.basecamp.com/2914079/buckets/28168307/vaults/5060979274).\n\n### Medical Insurance\n\nIn the United States, medical insurance is provided through Blue Cross Blue Shield PPO. The company pays 75% of the premium and the employee pays the other 25%. Open enrollment is in November every year, with new coverage beginning December 1. Marriages and domestic partnerships are covered. You’re eligible for coverage on your first day of employment. If you are terminated or resign from 37signals, your coverage will end on the last day of the month of your separation date and you may be eligible for continued coverage after that (COBRA).\n\nEach pay period, you’ll see a payroll deduction for medical insurance:\n\n* Employee-only medical coverage: $96.05\n* Employee-partner medical coverage: $197.27\n* Employee-child(ren) medical coverage: $

In [3]:
display(documents[0].page_content[:100])
print("Number of documents:", len(documents))

'# Benefits & Perks\n\n## Health Insurance\n\nDetailed information about all 37signals insurance policies'

Number of documents: 15


## Document Chunking

In [4]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=230,
    chunk_overlap=80
)

chunks = text_splitter.split_documents(documents)
print("Documents:", len(documents))
print("Chunks:", len(chunks))

Documents: 15
Chunks: 719


## Embedding

In [5]:

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="../vector_db"
)

print(
    "Vectors stored:",
    vector_store._collection.count()
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12921.76it/s]


Vectors stored: 3179


## Vector Store Retriever

In [6]:
retriever = vector_store.as_retriever(
    search_kwargs={"k": 5}
)

In [7]:
query = "How does 37signals handle employee benefits?"
retrieved_docs = retriever.invoke(query)
display(retrieved_docs)
print(len(retrieved_docs))

[Document(id='3d0d4ebd-1651-47ba-bc34-8ada94c7807f', metadata={'source': '..\\handbook\\README.md'}, page_content='# 37signals Employee Handbook\n\nIn this handbook, you’ll find everything you need to know about 37signals policies and benefits. Hopefully it also offers a small peek at our culture.'),
 Document(id='012c70e2-40d7-4fef-a24e-0274987d3873', metadata={'source': '..\\handbook\\stateFMLA.md'}, page_content='Below are states that have Paid Medical and Family Leave provisions and in which 37signals has employees. If you work in one of the following states and contribute to these programs, you may qualify'),
 Document(id='8e65792d-cbeb-4e05-a443-aa4e6393dcb1', metadata={'source': '..\\handbook\\benefits-and-perks.md'}, page_content='37signals provides MetLife Short Term and Long Term Disability policies to all US employees, at no cost to the employee. The Short Term policy may replace up to 70% of your salary, up to 12 weeks and'),
 Document(id='8fa7f7c9-ce8e-4999-9a67-de9f3c7010

5


In [8]:
query = "What are the benefits and perks at 37signals?"
retrieved_docs = retriever.invoke(query)

for i, doc in enumerate(retrieved_docs, 1):
    print(f"\n--- Result {i} ---")
    print(doc.page_content[:700])
    print("Source:", doc.metadata.get("source"))


--- Result 1 ---
# Benefits & Perks

## Health Insurance

Detailed information about all 37signals insurance policies and other benefits can be found in [Basecamp](https://3.basecamp.com/2914079/buckets/28168307/vaults/5060979274).
Source: ..\handbook\benefits-and-perks.md

--- Result 2 ---
# Benefits & Perks

## Health Insurance

Detailed information about all 37signals insurance policies and other benefits can be found in [Basecamp](https://3.basecamp.com/2914079/buckets/28168307/vaults/5060979274).
Source: ..\handbook\benefits-and-perks.md

--- Result 3 ---
# Benefits & Perks

## Health Insurance

Detailed information about all 37signals insurance policies and other benefits can be found in [Basecamp](https://3.basecamp.com/2914079/buckets/28168307/vaults/5060979274).
Source: ..\handbook\benefits-and-perks.md

--- Result 4 ---
Getting started at 37signals involves a lot of little details, a number of big tasks, learning the details of your new job, meeting new coworkers, all while 

In [9]:
for doc in retrieved_docs:
    print(doc.metadata["source"])

..\handbook\benefits-and-perks.md
..\handbook\benefits-and-perks.md
..\handbook\benefits-and-perks.md
..\handbook\getting-started.md
..\handbook\getting-started.md


In [10]:
results = vector_store.similarity_search_with_score(
    query,
    k=4
)

for i, (doc, score) in enumerate(results, 1):
    print(f"\n--- Result {i} ---")
    print("Score:", score)
    print("Source:", doc.metadata["source"])
    print(doc.page_content[:300])


--- Result 1 ---
Score: 0.6917320489883423
Source: ..\handbook\benefits-and-perks.md
# Benefits & Perks

## Health Insurance

Detailed information about all 37signals insurance policies and other benefits can be found in [Basecamp](https://3.basecamp.com/2914079/buckets/28168307/vaults/5060979274).

--- Result 2 ---
Score: 0.6917320489883423
Source: ..\handbook\benefits-and-perks.md
# Benefits & Perks

## Health Insurance

Detailed information about all 37signals insurance policies and other benefits can be found in [Basecamp](https://3.basecamp.com/2914079/buckets/28168307/vaults/5060979274).

--- Result 3 ---
Score: 0.6917320489883423
Source: ..\handbook\benefits-and-perks.md
# Benefits & Perks

## Health Insurance

Detailed information about all 37signals insurance policies and other benefits can be found in [Basecamp](https://3.basecamp.com/2914079/buckets/28168307/vaults/5060979274).

--- Result 4 ---
Score: 0.7806718349456787
Source: ..\handbook\getting-started.md
Getting starte

### Evaluate Retrieval Performance

In [11]:
eval_dataset = [
    # benefits-and-perks.md
    {
        "question": "What benefits and perks do employees receive?",
        "relevant_sources": ["benefits-and-perks.md"]
    },
    {
        "question": "What medical insurance is provided to employees?",
        "relevant_sources": ["benefits-and-perks.md"]
    },
    {
        "question": "What retirement plan is available to employees?",
        "relevant_sources": ["benefits-and-perks.md"]
    },
    {
        "question": "When does employee insurance coverage begin?",
        "relevant_sources": ["benefits-and-perks.md"]
    },
    {
        "question": "When does employee insurance coverage end?",
        "relevant_sources": ["benefits-and-perks.md"]
    },
    {
        "question": "Where can employees find more information about insurance policies?",
        "relevant_sources": ["benefits-and-perks.md"]
    },
    {
        "question": "What does the company provide for employee retirement?",
        "relevant_sources": ["benefits-and-perks.md"]
    },

    # making-a-career.md
    {
        "question": "How does 37signals handle employee careers?",
        "relevant_sources": ["making-a-career.md"]
    },
    {
        "question": "How is employee compensation determined?",
        "relevant_sources": ["making-a-career.md"]
    },
    {
        "question": "How often is compensation data reviewed?",
        "relevant_sources": ["making-a-career.md"]
    },
    {
        "question": "How does the company determine employee pay?",
        "relevant_sources": ["making-a-career.md"]
    },
    {
        "question": "What happens when market compensation rates increase?",
        "relevant_sources": ["making-a-career.md"]
    },
    {
        "question": "Does the company reduce salaries when market rates decrease?",
        "relevant_sources": ["making-a-career.md"]
    },

    # how-we-work.md
    {
        "question": "How does 37signals work?",
        "relevant_sources": ["how-we-work.md"]
    },
    {
        "question": "What are the company's approaches to working?",
        "relevant_sources": ["how-we-work.md"]
    },
    {
        "question": "How is work organized at 37signals?",
        "relevant_sources": ["how-we-work.md"]
    },
    {
        "question": "What principles guide how employees work?",
        "relevant_sources": ["how-we-work.md"]
    },

    # our-rituals.md
    {
        "question": "What are the rituals at 37signals?",
        "relevant_sources": ["our-rituals.md"]
    },
    {
        "question": "What recurring rituals does the company have?",
        "relevant_sources": ["our-rituals.md"]
    },
    {
        "question": "What meetings or rituals are part of company culture?",
        "relevant_sources": ["our-rituals.md"]
    },

    # our-internal-systems.md
    {
        "question": "What internal systems does 37signals use?",
        "relevant_sources": ["our-internal-systems.md"]
    },
    {
        "question": "Which systems are used internally by the company?",
        "relevant_sources": ["our-internal-systems.md"]
    },
    {
        "question": "What tools and systems are used by employees?",
        "relevant_sources": ["our-internal-systems.md"]
    },

    # severance.md
    {
        "question": "What is the severance policy?",
        "relevant_sources": ["severance.md"]
    },
    {
        "question": "What happens when an employee leaves the company?",
        "relevant_sources": ["severance.md"]
    },
    {
        "question": "What severance benefits are available to employees?",
        "relevant_sources": ["severance.md"]
    },

    # moonlighting.md
    {
        "question": "What is the company's policy on moonlighting?",
        "relevant_sources": ["moonlighting.md"]
    },
    {
        "question": "Can employees work on outside projects?",
        "relevant_sources": ["moonlighting.md"]
    },

    # README.md
    {
        "question": "What is the 37signals employee handbook about?",
        "relevant_sources": ["README.md"]
    },
    {
        "question": "What information is included in the employee handbook?",
        "relevant_sources": ["README.md"]
    },
]

## Optimize Chunking Parameters

In [12]:
best_result = {
    "chunk_size": 295,
    "chunk_overlap": 65,
    "k": 8
}

In [13]:
K = best_result["k"]

best_chunk_size = best_result["chunk_size"]
best_chunk_overlap = best_result["chunk_overlap"]

print(f"Chunk size   : {best_chunk_size}")
print(f"Chunk overlap: {best_chunk_overlap}")
print(f"K            : {K}")


text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=best_chunk_size,
    chunk_overlap=best_chunk_overlap
)

chunks = text_splitter.split_documents(documents)

print(f"Number of chunks: {len(chunks)}")



bge_embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5",
    model_kwargs={
        "device": "cpu"
    },
    encode_kwargs={
        "normalize_embeddings": True
    }
)

print("\ loaded.")



bge_vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=bge_embeddings,
    collection_name="bge_collection"
)

bge_retriever = bge_vector_store.as_retriever(
    search_kwargs={
        "k": K
    }
)

print("BGE Retriever ready.")




bm25_retriever = BM25Retriever.from_documents(
    chunks
)

bm25_retriever.k = K

print("BM25 Retriever ready.")


Chunk size   : 295
Chunk overlap: 65
K            : 8
Number of chunks: 531


<>:32: SyntaxWarning: invalid escape sequence '\ '
<>:32: SyntaxWarning: invalid escape sequence '\ '
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_14668\626651565.py:32: SyntaxWarning: invalid escape sequence '\ '
  print("\ loaded.")
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9983.45it/s]


\ loaded.
BGE Retriever ready.
BM25 Retriever ready.


In this step, we rebuild the document chunks using the best chunk size and overlap values found during the previous experiments. Then, we create **a BG**E embedding model to convert the text chunks into meaningful vector representations and store them in a Chroma vector database. Alongside the dense BGE retriever, we also set up **a BM25**retriever to support keyword-based search. Having both retrievers allows us to compare semantic search and traditional keyword retrieval approaches during the evaluation phase.

In [14]:
def evaluate_retriever(
    eval_dataset,
    retriever,
    k=K
):

    results = []

    for item in eval_dataset:

        question = item["question"]
        relevant_sources = item["relevant_sources"]

        retrieved_docs = retriever.invoke(question)

        retrieved_docs = retrieved_docs[:k]

        retrieved_sources = []

        for doc in retrieved_docs:

            source = doc.metadata.get(
                "source",
                ""
            )

            filename = (
                source
                .replace("\\", "/")
                .split("/")[-1]
            )

            retrieved_sources.append(filename)


        hit = any(
            source in relevant_sources
            for source in retrieved_sources
        )


        reciprocal_rank = 0.0

        for rank, source in enumerate(
            retrieved_sources,
            start=1
        ):

            if source in relevant_sources:

                reciprocal_rank = 1 / rank
                break


        relevant_retrieved = sum(
            source in relevant_sources
            for source in retrieved_sources
        )

        precision_at_k = (
            relevant_retrieved / k
        )


        relevant_found = len(
            set(retrieved_sources)
            &
            set(relevant_sources)
        )

        recall_at_k = (
            relevant_found /
            len(relevant_sources)
        )


        results.append({

            "question": question,
            "expected": relevant_sources,
            "retrieved": retrieved_sources,
            "hit": hit,
            "reciprocal_rank": reciprocal_rank,
            "precision_at_k": precision_at_k,
            "recall_at_k": recall_at_k
        })


    return results



def calculate_metrics(results):

    hit_rate = sum(
        result["hit"]
        for result in results
    ) / len(results)


    mrr = sum(
        result["reciprocal_rank"]
        for result in results
    ) / len(results)


    precision = sum(
        result["precision_at_k"]
        for result in results
    ) / len(results)


    recall = sum(
        result["recall_at_k"]
        for result in results
    ) / len(results)


    return {
        "Hit Rate": hit_rate,
        "MRR": mrr,
        "Precision@K": precision,
        "Recall@K": recall
    }



bge_results = evaluate_retriever(
    eval_dataset,
    bge_retriever,
    k=K
)


bm25_results = evaluate_retriever(
    eval_dataset,
    bm25_retriever,
    k=K
)


bge_metrics = calculate_metrics(
    bge_results
)


bm25_metrics = calculate_metrics(
    bm25_results
)



import pandas as pd

metrics_df = pd.DataFrame({

    "Metric": [
        "Hit Rate",
        "MRR",
        "Precision@K",
        "Recall@K"
    ],

    "BGE": [
        bge_metrics["Hit Rate"],
        bge_metrics["MRR"],
        bge_metrics["Precision@K"],
        bge_metrics["Recall@K"]
    ],

    "BM25": [
        bm25_metrics["Hit Rate"],
        bm25_metrics["MRR"],
        bm25_metrics["Precision@K"],
        bm25_metrics["Recall@K"]
    ]
})


metrics_df.style.format({
    "BGE": "{:.2%}",
    "BM25": "{:.2%}"
})

,Metric,BGE,BM25
0,Hit Rate,86.67%,70.00%
1,MRR,63.33%,47.70%
2,Precision@K,44.58%,24.17%
3,Recall@K,86.67%,70.00%


This function is used to evaluate the performance of a retriever on a given evaluation dataset. For each question, it retrieves the most relevant documents and compares the retrieved sources with the expected answers. It calculates several retrieval metrics, including Hit Rate, MRR, Precision@K, and Recall@K, to measure how effectively the retriever finds relevant information. The results are stored for further analysis and comparison between different retrieval methods.

The results show that the BGE retriever performs better than BM25 across all evaluation metrics. BGE achieves a higher Hit Rate and Recall, which means it is able to find the relevant documents for more questions. It also provides better Precision and MRR scores, indicating that the retrieved documents are not only more relevant but also appear in better positions within the ranked results. This improvement is expected because BGE uses semantic embeddings and can understand the meaning behind queries, while BM25 mainly relies on exact keyword matching. Overall, BGE provides a more effective retrieval approach for this dataset.

## Implement Hybrid Retrieval with Reciprocal Rank Fusion (RRF)

In [15]:
class HybridRRF:

    def __init__(
        self,
        retrievers,
        weights=None,
        k=60,
        top_k=3
    ):
        
        self.retrievers = retrievers
        self.weights = weights or [1] * len(retrievers)
        self.k = k
        self.top_k = top_k


    def invoke(self, query):

        all_results = []


        for retriever, weight in zip(
            self.retrievers,
            self.weights
        ):

            docs = retriever.invoke(query)


            for rank, doc in enumerate(
                docs,
                start=1
            ):

                score = weight * (
                    1 / (self.k + rank)
                )

                all_results.append(
                    {
                        "doc": doc,
                        "score": score
                    }
                )



        fused_scores = {}


        for item in all_results:

            content = item["doc"].page_content

            if content not in fused_scores:

                fused_scores[content] = {
                    "doc": item["doc"],
                    "score": 0
                }


            fused_scores[content]["score"] += item["score"]




        ranked_docs = sorted(
            fused_scores.values(),
            key=lambda x: x["score"],
            reverse=True
        )


        return [
            item["doc"]
            for item in ranked_docs[:self.top_k]
        ]

In this step, a hybrid retriever is implemented by combining multiple retrieval methods using Reciprocal Rank Fusion (RRF). The goal is to take advantage of both semantic and keyword-based search by merging their results into a single ranked list. Each retriever contributes scores based on the position of retrieved documents, and the final ranking is generated by combining these scores. This approach helps improve retrieval quality by balancing the strengths of different search strategies.

In [16]:
hybrid_retriever = HybridRRF(
    retrievers=[
        bge_retriever,
        bm25_retriever
    ],
    weights=[
        0.7,
        0.3
    ],
    top_k=K
)


print("Hybrid RRF ready")

Hybrid RRF ready


In [19]:
def calculate_metrics(
    results,
    k=K
):

    hit_rate = sum(
        result["hit"]
        for result in results
    ) / len(results)


    mrr = sum(
        result["reciprocal_rank"]
        for result in results
    ) / len(results)


    precision = sum(
        result["precision_at_k"]
        for result in results
    ) / len(results)


    recall = sum(
        result["recall_at_k"]
        for result in results
    ) / len(results)


    return {
        "Hit Rate": hit_rate,
        "Recall@K": recall
    }


hybrid_results = evaluate_retriever(
    eval_dataset,
    hybrid_retriever,
    k=K
)


hybrid_metrics = calculate_metrics(
    hybrid_results,
    K
)


print(hybrid_metrics)

{'Hit Rate': 0.8666666666666667, 'Recall@K': 0.8666666666666667}


By combining semantic search from BGE with keyword-based retrieval from BM25, the system benefits from both approaches. 

In [22]:
import os
import json
import pickle

from langchain_chroma import Chroma


# ============================================================
# Paths
# ============================================================

BASE_DIR = "artifacts"

CHROMA_DIR = os.path.join(
    BASE_DIR,
    "chroma_db"
)

BM25_PATH = os.path.join(
    BASE_DIR,
    "bm25_retriever.pkl"
)

CONFIG_PATH = os.path.join(
    BASE_DIR,
    "hybrid_config.json"
)


os.makedirs(
    BASE_DIR,
    exist_ok=True
)



# ============================================================
# Save Chroma Vector Store
# ============================================================

bge_vector_store = Chroma.from_documents(

    documents=chunks,

    embedding=bge_embeddings,

    collection_name="final_rag_collection",

    persist_directory=CHROMA_DIR

)


print("✅ Chroma vector store saved")



# ============================================================
# Save BM25 Retriever
# ============================================================

with open(
    BM25_PATH,
    "wb"
) as f:

    pickle.dump(
        bm25_retriever,
        f
    )


print("✅ BM25 retriever saved")



# ============================================================
# Save Hybrid Configuration
# ============================================================

hybrid_config = {

    "embedding_model":
        "BAAI/bge-base-en-v1.5",

    "chunk_size":
        best_result["chunk_size"],

    "chunk_overlap":
        best_result["chunk_overlap"],

    "K":
        K,

    "rrf_k":
        60,

    "weights":

    {
        "BGE": 0.7,
        "BM25": 0.3
    }

}



with open(
    CONFIG_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        hybrid_config,
        f,
        indent=4
    )


print("✅ Hybrid configuration saved")



# ============================================================
# Summary
# ============================================================

print("\nSaved Files:")
print("----------------")

for file in [

    CHROMA_DIR,

    BM25_PATH,

    CONFIG_PATH

]:

    print("📁", file)

✅ Chroma vector store saved
✅ BM25 retriever saved
✅ Hybrid configuration saved

Saved Files:
----------------
📁 artifacts\chroma_db
📁 artifacts\bm25_retriever.pkl
📁 artifacts\hybrid_config.json
